In [8]:
import shutil
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning

from input import input
import config
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# === CELULA DE CONFIGURACAO -- Tarefa 3.2 (fluxo notebook-only) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 3 do PLANO_ARQUITETURA.md: integracao do TimeSeriesFeatureSelector
# via sklearn.Pipeline. strategy fixa nesta execucao ('f_test' -- o quick win
# do roadmap); k varia como grid (selector__k abaixo). A transformacao de y
# (MinMaxScaler/KPSS) continua fora do Pipeline, dentro de
# Additive/generics.format_forecats (Secao 1.4 do PLANO_ARQUITETURA.md).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='f_test')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# austres.txt resolve para 1 unico lag via lag_size='auto' (RUNBOOK.md Secao 1) --
# avaliar se faz sentido incluir antes de comparar metodos de FS entre si.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
experiment_id = 'chamados_v4_fs_ftest'
# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py continuam funcionando sem nenhuma mudanca (ver Tarefa 3, Parte B).
model_name = 'amv1ftest'
normalize = True
force = True
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
model_parameters = {
    'selector__k': [1, 5, 9, 15, 20],   
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
}

In [10]:
# Sanity-check exigido pela Tarefa 2: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'selector__k', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

OK -- Pipeline.get_params(deep=True) expoe: ['estimator__hidden_layer_sizes', 'estimator__max_iter', 'selector__k', 'selector__strategy']


In [11]:
# Copia o ARIMA pre-treinado (mesmo modelo, nao retreina) para o novo
# experiment_id -- Additive precisa dele sob o MESMO experiment_id da variante
# FS (ver RUNBOOK.md Secao 3). Substitui o Passo 0 (bash) do RUNBOOK.md Secao
# 8b por uma celula Python equivalente -- Tarefa 3.2 (fluxo notebook-only).
arima_source_dir = Path(config.MODEL_DATA_PATH) / 'chamados'
arima_dest_dir = Path(config.MODEL_DATA_PATH) / experiment_id
arima_dest_dir.mkdir(parents=True, exist_ok=True)

for base_name in fs_series_list:
    serie = base_name.split('.')[0]
    src = arima_source_dir / f'{serie}_1arima.pkl'
    dst = arima_dest_dir / f'{serie}_1arima.pkl'
    shutil.copy(src, dst)
    print(f'{serie}: {src} -> {dst}')

airlines: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\airlines_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_ftest\airlines_1arima.pkl
austres: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\austres_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_ftest\austres_1arima.pkl
coloradoRiver: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\coloradoRiver_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_ftest\coloradoRiver_1arima.pkl
sunspot: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\sunspot_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_ftest\sunspot_1arima.pkl


In [12]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST e
# sem tocar grid_search_exp.py. use_val_slipt_for_prev=True e explicito porque
# o default de GridSearch (False) diverge do default do wrapper
# grid_seach_multiple_bases (True) -- sem isso, o refit final perderia o
# val_size e os .pkl ficariam sem val_metrics, quebrando a paridade com o
# baseline original (amv1 em chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

airlines.txt
{'estimator__hidden_layer_sizes': 20, 'estimator__max_iter': 1000, 'selector__k': 5}
austres.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 20}
coloradoRiver.txt
{'estimator__hidden_layer_sizes': 10, 'estimator__max_iter': 1000, 'selector__k': 9}
sunspot.txt
{'estimator__hidden_layer_sizes': 20, 'estimator__max_iter': 1000, 'selector__k': 5}


In [13]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (Tarefa 3.2, fluxo notebook-only). Equivale ao Passo 2 do
# RUNBOOK.md Secao 8b.
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = Path(config.ROOT_PATH) / 'results' / experiment_id / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    Path(config.MODEL_DATA_PATH) / experiment_id,
    metrics_output,
    detail=True,
)
df_metrics

[INFO] 8 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_ftest'.

  OK  airlines_1amv1ftest.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  austres_1amv1ftest.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1amv1ftest.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  sunspot_1amv1ftest.pkl  ->  10 linha(s)
  OK  sunspot_1arima.pkl  ->  1 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_ftest\metrics.csv
     8 linha(s) × 20 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_ftest\metrics_detail.csv
     44 linha(s) × 13 coluna(s)


,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_v4_fs_ftest,airlines,1amv1ftest,10,366.228976,1.033500,19.137093,0.027010,14.810528,0.023170,3.223259,0.005591,0.123837,0.000384,0.062447,0.000164,0.984142,0.000043,78.571429,0.000000
1,chamados_v4_fs_ftest,airlines,1arima,1,389.194049,NaN,19.728002,NaN,15.102607,NaN,3.315326,NaN,0.134713,NaN,0.066301,NaN,0.983139,NaN,78.571429,NaN
2,chamados_v4_fs_ftest,austres,1amv1ftest,10,365.457566,7.462870,19.116044,0.195540,14.055688,0.233106,0.080353,0.001326,0.131890,0.001887,0.035560,0.000643,0.990821,0.000178,75.000000,0.000000
3,chamados_v4_fs_ftest,austres,1arima,1,358.130880,NaN,18.924346,NaN,13.710292,NaN,0.078380,NaN,0.129713,NaN,0.034904,NaN,0.990999,NaN,75.000000,NaN
4,chamados_v4_fs_ftest,coloradoRiver,1amv1ftest,10,0.106548,0.004479,0.326353,0.006826,0.264549,0.005808,29.034075,0.615811,0.763039,0.020769,0.697652,0.010578,0.666271,0.004039,53.783784,3.107782
5,chamados_v4_fs_ftest,coloradoRiver,1arima,1,0.104961,NaN,0.323976,NaN,0.262984,NaN,28.817193,NaN,0.755655,NaN,0.698832,NaN,0.667249,NaN,52.702703,NaN
6,chamados_v4_fs_ftest,sunspot,1amv1ftest,10,365.018717,3.166571,19.105301,0.082951,15.619178,0.079953,39.734361,0.551642,0.332093,0.001412,0.201571,0.002741,0.950360,0.000583,60.714286,0.000000
7,chamados_v4_fs_ftest,sunspot,1arima,1,365.478709,NaN,19.117497,NaN,15.719171,NaN,36.503112,NaN,0.346894,NaN,0.193887,NaN,0.951367,NaN,67.857143,NaN


In [14]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (Tarefa 3.2, fluxo notebook-only). Equivale ao
# Passo 2 do RUNBOOK.md Secao 8b.
from utils.export_selected_features import run_export_selected_features

features_output = Path(config.ROOT_PATH) / 'results' / experiment_id / 'selected_features.csv'
df_features = run_export_selected_features(
    Path(config.MODEL_DATA_PATH) / experiment_id,
    features_output,
    detail=True,
)
df_features

[INFO] 8 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_ftest'.

  OK  airlines_1amv1ftest.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  sem seletor -- ignorado
  OK  austres_1amv1ftest.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  sem seletor -- ignorado
  OK  coloradoRiver_1amv1ftest.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  sem seletor -- ignorado
  OK  sunspot_1amv1ftest.pkl  ->  10 linha(s)
  OK  sunspot_1arima.pkl  ->  sem seletor -- ignorado

[OK] CSV agregado (média/desvio por série × modelo) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_ftest\selected_features.csv
     4 linha(s) × 8 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_ftest\selected_features_detail.csv
     40 linha(s) × 9 coluna(s)


,ExperimentID,Serie,Modelo,Strategy,N_Features_Selected_mean,N_Features_Selected_std,N_Repeticoes,N_Features_Total
0,chamados_v4_fs_ftest,airlines,1amv1ftest,f_test,5.0,0.0,10,20
1,chamados_v4_fs_ftest,austres,1amv1ftest,f_test,1.0,0.0,10,1
2,chamados_v4_fs_ftest,coloradoRiver,1amv1ftest,f_test,9.0,0.0,10,16
3,chamados_v4_fs_ftest,sunspot,1amv1ftest,f_test,5.0,0.0,10,9
